### Getting Started:
- Make sure you are outside the sam2 repository, then run `pip install -r requirements.txt`

In [1]:
import torch
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import matplotlib.pyplot as plt
import cv2
import pandas as pd

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [3]:
def show_anns(anns, borders=True, path='example_image.png'):
    """
    NOTE: This function assumes you are saving to a folder called 'output_masks' 
    in the parent dir.
    """
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)
 
    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    # print("Initial canvas:", np.array(img))
    
    # Set default pixel transparency to ... (0 for transparent, 1 for full)
    img[:,:,3] = 1 

    # At this point, sorted_anns is a list of the segmented masks SAM2 found in the data
    for ann in sorted_anns:
        # For every area in the segmentation, get the mask at anns['segmentation']...
        m = ann['segmentation']
        
        # Use a color (that is NOT white)
        color_mask = np.concatenate([np.random.uniform(low=0.1, high=1, size=(3)), [1]])
        img[m] = color_mask 
        # and plot the colors
        if borders:
            contours, _ = cv2.findContours(m.astype(np.uint8),cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0,0,1,0.4), thickness=1) 
 

    ax.imshow(img)
    # print("Anns mask", sorted_anns[0]['segmentation'].shape)
    # print("Image as array:", np.array(img))

    

    plt.imsave(f"../output_masks/{path}", img)

In [4]:
def noisify_image(image:np.array, noise:str="random", threshold=0.1):
    # add noise to image
    # snow is generally greyscale, so noise values should be in that range
        # uint8 dtype limits cv2 randn to [0, 255]
    noise_mask = np.zeros(shape=(image.shape[0], image.shape[1], 3), dtype=np.uint8)

    if noise == "gaussian":
        # apply gaussian noise
        cv2.randn(noise_mask, mean=(128, 128, 128), stddev=(40, 40, 40))
        noise_mask = (noise_mask * 0.5).astype(np.uint8) # dilute the noise so that its application to the iamge is more realistic
        image = np.add(image, noise_mask)

    # add more noises methods here...
    elif noise == "random":
        for i in range(image.shape[0]):
            for j in range(image.shape[1]):
                if np.random.random() <= threshold:
                    image[i][j] = (np.random.rand(3) * 255).astype(np.uint32)
        

    return image

In [5]:
checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
mask_generator = SAM2AutomaticMaskGenerator(build_sam2(model_cfg, checkpoint, device=device, apply_postprocessing=False))

In [ ]:
image=Image.open("../0001TP_009240.png")
image=np.array(image.convert("RGB"))

print("Before Noise:")
plt.figure(figsize=(20, 20))
plt.imshow(image)
plt.axis('off')
plt.show()

# thresholds 0.3 and above are a bit brutal - like a snow day snowstorm.
image = noisify_image(image, noise="random", threshold=0.1)

print("After Noise:")
plt.figure(figsize=(20, 20))
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
masks = mask_generator.generate(image)

plt.figure(figsize=(20,20))
plt.imshow(image)
show_anns(masks)
plt.axis('off')
plt.show() 

In [ ]:
# Getting individual masks from SAM2

# Example image - clean, no noise
image=Image.open("../0001TP_009240.png")
image=np.array(image.convert("RGB"))

masks = mask_generator.generate(image)
for key in masks[0]: # assume sam2 has 1 or more masks
    print(key, ":", masks[0][key])

print("--------------------------------------------")
print("SAM2 Output Mask example shape:", masks[0]['segmentation'].shape)
print("Input image shape:", image.shape)

print("--------------------------------------------")
first_component = image
for i in range(first_component.shape[0]):
    for j in range(first_component.shape[1]):
        if not masks[0]['segmentation'][i][j]:
            first_component[i][j] = [0,0,0] 

plt.figure(figsize=(20, 20))
plt.imshow(first_component)
plt.axis('off')
plt.show()

## Common Bugs:
- "Torch is not compiled with Cuda" - uninstall torch, torchvision, torchaudio (`pip uninstall torch torchvision torchaudio`) and install the newest versions of each w/ Cuda (command available on Pytorch website). 
    - ATM this is:
     ```pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124```
- "Not implemented error" - this issue occurs when the device is Cuda but the Cuda version is not compatible with the GPU. Again, same fix as above (essentially update your Cuda version)
- "Commit denied by prereceive hook" - this issue occurs when the notebook is run & committed with all output. The output images in the notebook cause it to grow beyond git's acceptable size (100-120 mb). To fix, clear all output and restart kernel before committing.

In [9]:
import os
import random

dataset_path = "../download_dataset"
output_path = "../output_masks/val"

val_dir = os.path.join(dataset_path, 'val')

In [10]:
k = 0 # to see effects, set k > 0
val_files = os.listdir(val_dir)
noise="random" # Disable by setting to None
show_img = True

for file in random.choices(val_files, k=k):
    try:
        src = os.path.join(val_dir, file)
        
        # Get original image
        image=Image.open(src)
        image=np.array(image.convert("RGB"))
        
        if show_img:
            plt.figure(figsize=(20, 20))
            plt.imshow(image)
            plt.axis('off')
            plt.show()
            print("Original Image")

        if noise:
            image = noisify_image(image)
            
            if show_img:
                print("After Noise:")
                plt.figure(figsize=(20, 20))
                plt.imshow(image)
                plt.axis('off')
                plt.show()

        # Show SAM2 Output
        masks = mask_generator.generate(image)
        plt.figure(figsize=(20,20))
        plt.imshow(image)
        show_anns(masks, path=f"val/{file}")
        plt.axis('off')
        plt.show() 
        print("SAM2 Segmented Output")

        # Show Ground Truth
        file_labelled = file[:-4] + "_L" + file[-4:] # labelled files have a '_L' in them
        src_labelled = os.path.join(dataset_path, 'val_labels')
        src_labelled = os.path.join(src_labelled, file_labelled) # Get the related labeled image filepath
        image=Image.open(src_labelled)
        image=np.array(image.convert("RGB"))
        plt.figure(figsize=(20, 20))
        plt.imshow(image)
        plt.axis('off')
        plt.show()
        print("Ground Truth")
    except:
        print("An image could not be found. Moving on...")

In [ ]:
# Testing SAM2's resilience to noise
max_times_ten=0 # set to 5 to see full range

found_image = False
while not found_image:
    try:
        file = random.choice(val_files)
        src = os.path.join(val_dir, file)    
        # Get original image
        image=Image.open(src)
        image=np.array(image.convert("RGB"))
        found_image = True
    except:
        print("Could not find an image. Trying another ...")

# Test SAM2's resilience to noise
for i in range(1, max_times_ten+1):
    t = i / 10
    image = noisify_image(image, threshold=t)
    
    print(f"Noisy Image at threshold {i / 10}:")
    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    # Show SAM2 Output
    masks = mask_generator.generate(image)
    plt.figure(figsize=(20,20))
    plt.imshow(image)
    show_anns(masks, path=f"val/threshold_{t}_{file}")
    plt.axis('off')
    plt.show() 
    print("SAM2 Segmented Output")

# Show Ground Truth
try:
    file_labelled = file[:-4] + "_L" + file[-4:] # labelled files have a '_L' in them
    src_labelled = os.path.join(dataset_path, 'val_labels')
    src_labelled = os.path.join(src_labelled, file_labelled) # Get the related labeled image filepath
    image=Image.open(src_labelled)
    image=np.array(image.convert("RGB"))
    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    print("Ground Truth")
except:
    print("Could not find ground truth. Moving on...")

<h2>YOLO & SAM2</h2>

In [ ]:
%pip install ultralytics

In [ ]:
table = pd.read_csv("../download_dataset/class_dict.csv")
# print(table.head())

# dict based on english labels
class_dict = {row["name"].lower() : [row["r"] / 255, row["g"] / 255, row["b"] / 255] for _, row in table.iterrows()}
color_labels = [class_dict[key] for key in class_dict]

print("Class dict:")
print(class_dict)
label_map = {
    'person':'pedestrian'
}

def label_to_color(label:str):
    if label in class_dict:
            color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
    else:
        if label in label_map:
            label = label_map[label]
            if label in class_dict:
                color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
            else:
                print("Could not find", label)
                color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
        else:
            print("Could not find", label)
            color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
    return color

In [14]:
def show_mask(mask, ax, label=None, index=-1):
    # edit so that color aligns w/ camvid class dict

    if label:
        color = label_to_color(label)
    else:
        if index > -1 and index < len(color_labels):
            color = np.concatenate([np.array(color_labels[index]), np.array([1])], axis=0)
        else:
            color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
    
    
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   
    
def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))    

### MASKS ARE 720 * 960 boolean masks of the image

In [15]:
# bbox = np.reshape(bbox, shape=(-1,1)).shape
from typing import List

def SAM2_segment_box(image, predictor, bbox, labels=None)->List:
    predictor.set_image(image=image)
    plt.figure(figsize=(10, 10))
    plt.imshow(image)

    n = len(bbox)
    # print(n)

    object_mask_dict = {}
    list_of_masks = [] # final length should match n
    for i in range(n):
        obj_detected, label = bbox[i], labels[i]

        input_box = np.array(obj_detected)

        masks, _, _ = predictor.predict(
            point_coords=None,
            point_labels=None,
            box=obj_detected,
            multimask_output=False
        )
        
        # print("Mask format:")
        # print(np.shape(masks[0])) # 720 X  960
        # print(np.max(masks[0])) # 1.0
        show_mask(masks[0], plt.gca(), label=label)
        show_box(input_box, plt.gca())

        if label in object_mask_dict:
            # union the two masks
            object_mask_dict[label] = np.logical_or(masks[0], object_mask_dict[label])
        else:
            object_mask_dict[label] = masks[0]
        list_of_masks.append(masks[0])
    plt.axis('off')
    plt.show()
    return object_mask_dict, list_of_masks

In [ ]:
from ultralytics import YOLO

img_path = "../0001TP_009240.png"

yolo_model = YOLO('yolov8n.pt')

sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
results = yolo_model.predict(source=img_path, conf=0.25)
names = results[0].names

for result in results: # Usually results is of length 1 - can change
    boxes = result.boxes
    bbox = boxes.xyxy.tolist() # bounding boxes AKA areas where YOLO found an object
    
    # print(boxes.cls)
    # print(result.names)

    image=Image.open(img_path)
    image=np.array(image.convert("RGB"))

    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    object_mask_dict, list_of_masks = SAM2_segment_box(image=image, predictor=predictor, bbox=bbox, labels = [names[index] for index in boxes.cls.tolist()])
    print(f"Found {len(list_of_masks)} objects in image.") 

In [ ]:
# Verify that we work
print(image.shape[1])
for label in object_mask_dict:
    mask_overall = object_mask_dict[label]
    color = label_to_color(label)[:-1]
    canvas = np.ones((720, 960, 3), dtype=np.float32)
    
    # print(mask_overall.shape)
    # print(canvas.shape)


    for i in range(mask_overall.shape[0]):
        for j in range(mask_overall.shape[1]):
            if mask_overall[i][j]:
                # print(canvas[i][j])
                # print(color)
                canvas[i][j] = color

    plt.figure(figsize=(10, 10))
    plt.imshow(canvas)    
    plt.axis('off')
    plt.show()

<h1>Fine-Tuning SAM2 w/ CamVid</h1>

Credits:
- [Video](https://www.youtube.com/watch?v=bcwLbmALyLI)
- [Medium Article](https://medium.com/towards-data-science/train-fine-tune-segment-anything-2-sam-2-in-60-lines-of-code-928dd29a63b3)
- [Repository](https://github.com/sagieppel/fine-tune-train_segment_anything_2_in_60_lines_of_code/tree/main)

In [1]:
import numpy as np
import torch
import cv2
import os
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [2]:
data_dir = "../CamVid/"
data = []

# ff=index, name=filename
for ff, name in enumerate(os.listdir(data_dir + "train/")):
    try:
        data.append({
            "image":data_dir + "train/"+name,
            "annotation":data_dir+"train_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")

for pair in data[:5]:
    print(pair["image"], pair["annotation"]) # to test above works

../CamVid/train/0001TP_009210.png ../CamVid/train_labels/0001TP_009210_L.png
../CamVid/train/0001TP_009240.png ../CamVid/train_labels/0001TP_009240_L.png
../CamVid/train/0001TP_009390.png ../CamVid/train_labels/0001TP_009390_L.png
../CamVid/train/0001TP_009420.png ../CamVid/train_labels/0001TP_009420_L.png
../CamVid/train/0001TP_009450.png ../CamVid/train_labels/0001TP_009450_L.png


In [3]:
def read_batch(data): 
    # read random image and its annotatio from the CamVid dataset

    entry = data[np.random.randint(len(data))] # Choose a random entry
    img = cv2.imread(entry["image"])
    ann_map = cv2.imread(entry["annotation"])

    # Normally we'd resize here. However, images are smaller than 1024 in both dimensions.
    # Thus, we skip resizing.

    # Fill in missing data
    mat_map = ann_map[:,:,0] # material map
    ves_map = ann_map[:,:,2] # vessel map
    mat_map[mat_map == 0] = ves_map[mat_map == 0] * (mat_map.max() + 1) # merging the two maps together 

    inds = np.unique(mat_map)[1:] # load all indices
    points = []
    masks = []

    # Use unique indices to sort pixels into masks
    for ind in inds:
        mask = (mat_map == ind).astype(np.uint8) # make binary mask
        masks.append(mask)
        coords = np.argwhere(mask > 0) # get all coordinates in mask
        yx = np.array(coords[np.random.randint(len(coords))]) # choose random point/coordinate from mask
        points.append([[yx[1], yx[0]]]) # x,y

    return img, np.array(masks), np.array(points), np.ones([len(masks), 1])


In [4]:
sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda") # load mmodel
predictor = SAM2ImagePredictor(sam2_model) # load net

In [5]:
predictor.model.sam_mask_decoder.train(True) # enable training of mask decoder
predictor.model.sam_prompt_encoder.train(True) # enable training of prompt decoder

PromptEncoder(
  (pe_layer): PositionEmbeddingRandom()
  (point_embeddings): ModuleList(
    (0-3): 4 x Embedding(1, 256)
  )
  (not_a_point_embed): Embedding(1, 256)
  (mask_downscaling): Sequential(
    (0): Conv2d(1, 4, kernel_size=(2, 2), stride=(2, 2))
    (1): LayerNorm2d()
    (2): GELU(approximate='none')
    (3): Conv2d(4, 16, kernel_size=(2, 2), stride=(2, 2))
    (4): LayerNorm2d()
    (5): GELU(approximate='none')
    (6): Conv2d(16, 256, kernel_size=(1, 1), stride=(1, 1))
  )
  (no_mask_embed): Embedding(1, 256)
)

In [6]:
optimizer = torch.optim.AdamW(
    params=predictor.model.parameters(), 
    lr=1e-5,
    weight_decay=4e-5
)
scaler = torch.cuda.amp.GradScaler()

C:\Users\Py Torch\AppData\Local\Temp\ipykernel_237108\2371007806.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [7]:
# training loop
from datetime import datetime

for itr in range(1, 1001):
    with torch.cuda.amp.autocast(): # cast to mix precision

        image, mask, input_point, input_label = read_batch(data)
        if mask.shape[0] == 0: continue # skip empty batches
        predictor.set_image(image) # apply SAM2 image encoder to training image

        # prompt encoding
        mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
        sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

        # mask decoder
        batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
        high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]
        low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(image_embeddings=predictor._features["image_embed"][-1].unsqueeze(0),image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),sparse_prompt_embeddings=sparse_embeddings,dense_prompt_embeddings=dense_embeddings,multimask_output=True,repeat_image=batched_mode,high_res_features=high_res_features,)
        prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])# Upscale the masks to the original image resolution

        # segmentation loss calculation
        gt_mask = torch.tensor(mask.astype(np.float32)).cuda()
        prd_mask = torch.sigmoid(prd_masks[:, 0])# Turn logit map to probability map
        seg_loss = (-gt_mask * torch.log(prd_mask + 0.00001) - (1 - gt_mask) * torch.log((1 - prd_mask) + 0.00001)).mean() # cross entropy loss
    
        # score loss using IOU
        inter = (gt_mask * (prd_mask > 0.5)).sum(1).sum(1)
        iou = inter / (gt_mask.sum(1).sum(1) + (prd_mask > 0.5).sum(1).sum(1) - inter)
        score_loss = torch.abs(prd_scores[:, 0] - iou).mean()
        loss=seg_loss+score_loss*0.05  # mix losses

        # backpropogate loss
        predictor.model.zero_grad() # empty gradient
        scaler.scale(loss).backward()  # Backpropogate
        scaler.step(optimizer)
        scaler.update() # Mix precision
        
        # Save model
        if itr%1000==0: 
            date_str = str(datetime.now()).replace(":","-")
            torch.save(predictor.model.state_dict(), f"../models/model-{date_str}.torch")
            print("saved model.")
    
        # Display results
        if itr==1: mean_iou=0
        mean_iou = mean_iou * 0.99 + 0.01 * np.mean(iou.cpu().detach().numpy())
        print("step)",itr, "Accuracy(IOU)=",mean_iou)


C:\Users\Py Torch\AppData\Local\Temp\ipykernel_237108\487114594.py:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): # cast to mix precision


step) 1 Accuracy(IOU)= 0.0005501105
step) 2 Accuracy(IOU)= 0.0013184163
step) 3 Accuracy(IOU)= 0.0013748734
step) 4 Accuracy(IOU)= 0.0017487935
step) 5 Accuracy(IOU)= 0.0020649352
step) 6 Accuracy(IOU)= 0.0021767963
step) 7 Accuracy(IOU)= 0.0032836325
step) 8 Accuracy(IOU)= 0.004048585
step) 9 Accuracy(IOU)= 0.0063900556
step) 10 Accuracy(IOU)= 0.007498106
step) 11 Accuracy(IOU)= 0.007796824
step) 12 Accuracy(IOU)= 0.008665562
step) 13 Accuracy(IOU)= 0.008903346
step) 14 Accuracy(IOU)= 0.009698453
step) 15 Accuracy(IOU)= 0.009859309
step) 16 Accuracy(IOU)= 0.010389948
step) 17 Accuracy(IOU)= 0.010671016
step) 18 Accuracy(IOU)= 0.011592784
step) 19 Accuracy(IOU)= 0.011831187
step) 20 Accuracy(IOU)= 0.012072691
step) 21 Accuracy(IOU)= 0.013080841
step) 22 Accuracy(IOU)= 0.013969085
step) 23 Accuracy(IOU)= 0.01611736
step) 24 Accuracy(IOU)= 0.016542552
step) 25 Accuracy(IOU)= 0.018365998
step) 26 Accuracy(IOU)= 0.020066392
step) 27 Accuracy(IOU)= 0.020597665
step) 28 Accuracy(IOU)= 0.0213

In [8]:
# Validate our finetuned SAM2

# use bfloat16 for memory efficiency
torch.autocast(device_type="cuda", dtype=torch.float32).__enter__()

# load some example images
val = []
for ff, name in enumerate(os.listdir(data_dir + "val/")[:5]):
    try:
        val.append({
            "image":data_dir + "val/"+name,
            "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")


for pair in val:
    print(pair["image"], pair["annotation"]) # to test above works

../CamVid/val/0001TP_009030.png ../CamVid/val_labels/0001TP_009030_L.png
../CamVid/val/0001TP_009060.png ../CamVid/val_labels/0001TP_009060_L.png
../CamVid/val/0001TP_009270.png ../CamVid/val_labels/0001TP_009270_L.png
../CamVid/val/0001TP_009300.png ../CamVid/val_labels/0001TP_009300_L.png
../CamVid/val/0001TP_009870.png ../CamVid/val_labels/0001TP_009870_L.png


In [9]:
def read_image(image_path, mask_path):
    # read an image and its mask
    image = cv2.imread(image_path)[...,::-1] # convert bgr to rgb
    mask = cv2.imread(mask_path, 0) # load masks in grayscale

    # Again, no resizing since images are less than 1024px on both dimensions

    return image, mask

def get_points(mask, num_points): # Sample points inside the input mask
    points = []
    for i in range(num_points):
        coords = np.argwhere(mask > 0)
        yx = np.array(coords[np.random.randint(len(coords))])
        points.append([[yx[1], yx[0]]])
    return np.array(points)

In [10]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)
input_points = get_points(mask, num_points=30) # arbitrarily get 30 points

In [11]:
model_dir = "../models/"

# build finetuned model and load weights
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load(model_dir + os.listdir(model_dir)[-1]))

C:\Users\Py Torch\AppData\Local\Temp\ipykernel_237108\1019397167.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  predictor.model.load_state_dict(torch.load(model_dir + o

<All keys matched successfully>

In [12]:
# predict masks
with torch.no_grad():
    predictor.set_image(image.copy())
    masks, scores, logits = predictor.predict(
        point_coords=input_points,
        point_labels=np.ones([input_points.shape[0], 1])
    )

# short predicted masks from high to low score
# essentially order masks by confidence scorwa
np_masks = np.array(masks[:,0]) 
np_scores = scores[:,0]
shorted_masks = np_masks[np.argsort(np_scores)][::-1]

# create empty segmentation map and occupancy map
# then, add masks to segmentation map one by one
# we only add a mask if it's consistent with previously added masks
    # AKA less tha 15% overlap with already occupied areas
seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)
for i in range(shorted_masks.shape[0]):
    mask = shorted_masks[i]
    if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
    mask[occupancy_mask]=0
    
    # Convert mask to boolean for idnexing
    seg_map[mask.astype(np.uint8)]=i+1
    occupancy_mask[mask > 0.5]=1

# create colored annotation map
height, width = seg_map.shape

# create empty rgb image for colored annotation
rgb_image = np.zeros([height, width, 3], dtype=np.uint8)

# map each class to a random color 
# later, convert this to camvid coloring
for id_class in range(1, seg_map.max() + 1):
    rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

In [13]:
# display image
cv2.imshow("annotation", rgb_image)
cv2.imshow("mix", (rgb_image / 2 + image / 2).astype(np.uint8))
cv2.imshow("image", image)
cv2.waitKey()

-1

# SAM2, then YOLO
SAM2 segment for masks, YOLO label masks